# REVELA WLC Operational Priority Score (OPS) — Weight Sensitivity Analysis

This notebook tests how **barangay priority rankings** change when the three OPS weights are varied. It uses the production OPS rules exactly as specified.

**What you do**

1. Run the first code cell and upload **one CSV or Excel (.xlsx)** file.
2. Check the detected column names and map them to the required fields.
3. Fix any validation errors the notebook reports.
4. Run the remaining cells. The notebook computes the production baseline, checks parity if production OPS/rank columns exist, then evaluates the full 231-combination weight grid plus the exact baseline.
5. Download the CSV and Excel results.

**Where the CSV comes from**

Use `export_ops_snapshot.py` in this folder. It reads the database in a read-only, consistent-snapshot transaction, runs the production effective-colour SQL and scoring function, and writes `ops_sensitivity_input.csv` with `production_ops` and `production_rank` already included for the parity check.

**Important**

- The notebook does **not** invent data, weights, AHP values, or results.
- If required production inputs are missing, it **stops** and lists what is missing.
- The full municipal cohort stays fixed. Dashboard filters are **not** applied and the cohort is **not** recomputed per scenario.
- This measures **ranking robustness under changed weights**. It does **not** prove predictive accuracy or better inspection outcomes.

## Data you must upload

Upload **one row per barangay** for the **full municipal cohort**. The file should be a production export, not a hand-filtered spreadsheet.

### Required columns

| Canonical field | Meaning |
|---|---|
| `barangay_id` | Unique barangay ID used by production |
| `barangay_name` | Barangay name |
| `red_count` | Distinct eligible geospatial records whose **effective** colour is Red |
| `yellow_count` | Distinct eligible geospatial records whose **effective** colour is Yellow |
| `black_count` | Distinct eligible geospatial records whose **effective** colour is Black. Keep this column even when the value is 0. |
| `avg_lat` | Mean latitude of valid, **dispatchable** effective Red/Yellow records in the barangay. This is **not** the barangay centroid. |
| `avg_lng` | Mean longitude of the same valid, dispatchable effective Red/Yellow records |

### Sector score

Supply `sector_score` (0–100, configured barangay score) **or** explicitly set `SECTOR_OVERRIDES_ABSENT = True` only if you have confirmed that no barangay has an override. Missing overrides use the production neutral score of **50**.

### BPLO coordinates

Supply `bplo_lat` and `bplo_lng` columns, **or** enter the production values as `BPLO_LAT` and `BPLO_LNG` in the mapping cell. Do not guess them.

### Optional columns

- `production_ops` and `production_rank`: both are needed if you want the parity check. If supplied, the notebook compares the baseline with production for every barangay and stops the sensitivity analysis when the maximum absolute OPS difference is greater than 0.1.
- `dispatchable_red_count`, `dispatchable_yellow_count`, `dispatchable_business_count`, `open_inspection_excluded`: all four are needed for the dispatch-eligible top-three comparison. `open_inspection_excluded` is `True` when the barangay is excluded by an open Assigned, Reassigned, or In Progress inspection report.
- `flagged_count`: optional cross-check. It must equal `red_count + yellow_count + black_count`.
- `sector_score_source`: optional label such as `configured` or `neutral_fallback`.

### Export rules you must verify

- Counts are **distinct eligible geospatial records**, not unique businesses.
- Colours must be the **production effective colours**, not a manually simplified `WHERE flagColor IN (...)` filter.
- The export must follow the production eligibility logic for actionable pins and dispatchable records.
- Do **not** apply dashboard filters. The cohort must be the full municipality.
- Remove business names, addresses, and other personal information. This notebook only needs barangay-level aggregates.

If a required value is missing, the notebook stops. It does not fill gaps with assumptions.

In [ ]:
# ============================================================
# STEP 1 — Upload your CSV or Excel file
# ============================================================
import os
import re
import math
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

try:
    from IPython.display import display, HTML
except Exception:
    def display(value):
        print(value)
    def HTML(value):
        return value

warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# If your Excel workbook has more than one sheet, put the exact sheet name here.
EXCEL_SHEET = None

# Local/automated runs can set REVELA_OPS_DATA to a file path instead of uploading.
LOCAL_DATA_PATH = os.environ.get("REVELA_OPS_DATA", "").strip()

if LOCAL_DATA_PATH:
    DATA_PATH = Path(LOCAL_DATA_PATH).expanduser().resolve()
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"REVELA_OPS_DATA points to a file that does not exist: {DATA_PATH}"
        )
    print(f"Using local data file: {DATA_PATH}")
elif IN_COLAB:
    from google.colab import files

    print("Upload ONE CSV or Excel (.xlsx) file containing your barangay-level production export.")
    print("The file should have one row per barangay in the full municipal cohort.")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file was uploaded. Re-run this cell and choose a file.")
    if len(uploaded) != 1:
        raise ValueError(
            "Please upload exactly one file. Re-run this cell and choose one CSV or .xlsx file."
        )

    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    suffix = Path(uploaded_name).suffix.lower()
    if suffix not in {".csv", ".xlsx"}:
        raise ValueError(
            f"Unsupported file type: {suffix}. Upload a .csv or .xlsx file. "
            "If you have an old .xls file, save it as .xlsx or CSV first."
        )

    DATA_PATH = Path("/content") / uploaded_name
    DATA_PATH.write_bytes(uploaded_bytes)
    print(f"Saved upload to: {DATA_PATH} ({len(uploaded_bytes):,} bytes)")
else:
    raise FileNotFoundError(
        "This notebook is not running in Google Colab and REVELA_OPS_DATA is not set. "
        "Open it in Colab to use the upload widget, or set REVELA_OPS_DATA to a local CSV/XLSX path."
    )

print("Step 1 complete.")

In [ ]:
# ============================================================
# STEP 2 — Load the file and show what was detected
# ============================================================
suffix = DATA_PATH.suffix.lower()

if suffix == ".csv":
    read_error = None
    for encoding in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            raw_df = pd.read_csv(
                DATA_PATH, sep=None, engine="python", encoding=encoding
            )
            print(f"Read CSV with encoding: {encoding}")
            break
        except Exception as exc:
            read_error = exc
    else:
        raise ValueError(f"Could not read the CSV file. Last error: {read_error}")
elif suffix == ".xlsx":
    workbook = pd.ExcelFile(DATA_PATH, engine="openpyxl")
    if EXCEL_SHEET is None:
        if len(workbook.sheet_names) == 1:
            sheet_name = workbook.sheet_names[0]
        else:
            print("Sheets found:", workbook.sheet_names)
            raise ValueError(
                "This workbook has more than one sheet. Set EXCEL_SHEET in the upload cell "
                "to the exact sheet name you want, then re-run the upload and load cells."
            )
    else:
        sheet_name = EXCEL_SHEET
        if sheet_name not in workbook.sheet_names:
            raise ValueError(
                f"EXCEL_SHEET={sheet_name!r} was not found. Sheets: {workbook.sheet_names}"
            )
    raw_df = pd.read_excel(workbook, sheet_name=sheet_name, engine="openpyxl")
    print(f"Read Excel sheet: {sheet_name}")
else:
    raise ValueError(f"Unsupported file type: {suffix}")

raw_df.columns = [str(column).strip() for column in raw_df.columns]

empty_rows = int(raw_df.isna().all(axis=1).sum())
if empty_rows:
    raw_df = raw_df.loc[~raw_df.isna().all(axis=1)].copy()
    print(f"Dropped {empty_rows} completely empty row(s).")

if len(raw_df) == 0:
    raise ValueError("The file contains no data rows. Nothing can be analysed.")
if len(raw_df.columns) == 0:
    raise ValueError("The file contains no columns. Nothing can be analysed.")

print(f"\nRows: {len(raw_df):,}")
print(f"Columns: {len(raw_df.columns):,}")
print("\nDetected column names:")
for name in raw_df.columns:
    print(f"  - {name}")

print("\nFirst five rows (check that this is the right file and the right sheet):")
display(raw_df.head())

## Map your columns

The notebook auto-detects common column names. Check the table it prints.

- If a required field says **NOT FOUND**, uncomment the matching `COLUMN_MAP[...]` line in the next cell and type the exact column name.
- If a field is mapped to the wrong column, override it the same way.
- Re-run the mapping cell after editing, then run the validation cell.

Do not map a barangay centroid column to `avg_lat`/`avg_lng`. Those fields must be the mean coordinates of valid, **dispatchable effective Red/Yellow records**.

In [ ]:
# ============================================================
# STEP 3 — Map your columns to the fields the notebook needs
# ============================================================
# After editing COLUMN_MAP, re-run this cell, then run the validation cell.


def normalise_name(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")


ALIASES = {
    "barangay_id": ["barangay_id", "barangayid", "brgy_id", "brgyid", "barangay_code"],
    "barangay_name": ["barangay_name", "barangayname", "brgy_name", "brgyname", "barangay"],
    "red_count": ["red_count", "redcount", "count_red", "effective_red_count", "red_records"],
    "yellow_count": ["yellow_count", "yellowcount", "count_yellow", "effective_yellow_count", "yellow_records"],
    "black_count": ["black_count", "blackcount", "count_black", "effective_black_count", "black_records"],
    "flagged_count": ["flagged_count", "flaggedcount", "total_flagged", "active_flag_count", "total_flags"],
    "sector_score": ["sector_score", "sectorscore", "configured_sector_score", "barangay_sector_score", "sector"],
    "sector_score_source": ["sector_score_source", "sector_source", "sector_status"],
    "avg_lat": ["avg_lat", "mean_lat", "average_latitude", "mean_latitude", "dispatchable_mean_lat"],
    "avg_lng": ["avg_lng", "mean_lng", "average_longitude", "mean_longitude", "dispatchable_mean_lng", "avg_lon", "mean_lon"],
    "bplo_lat": ["bplo_lat", "bplo_latitude", "office_lat", "office_latitude"],
    "bplo_lng": ["bplo_lng", "bplo_longitude", "office_lng", "office_longitude", "bplo_lon", "office_lon"],
    "production_ops": ["production_ops", "production_ops_score", "ops_score", "ops", "baseline_ops"],
    "production_rank": ["production_rank", "ops_rank", "priority_rank", "rank"],
    "dispatchable_red_count": ["dispatchable_red_count", "dispatch_red_count", "red_dispatchable_count"],
    "dispatchable_yellow_count": ["dispatchable_yellow_count", "dispatch_yellow_count", "yellow_dispatchable_count"],
    "dispatchable_business_count": ["dispatchable_business_count", "dispatchable_businesses", "eligible_business_count", "dispatchable_business"],
    "open_inspection_excluded": ["open_inspection_excluded", "has_open_inspection", "excluded_by_open_inspection", "open_report_excluded"],
}

REQUIRED_FIELDS = [
    "barangay_id",
    "barangay_name",
    "red_count",
    "yellow_count",
    "black_count",
    "avg_lat",
    "avg_lng",
]
OPTIONAL_FIELDS = [field for field in ALIASES if field not in REQUIRED_FIELDS]

norm_to_actual = {}
for column in raw_df.columns:
    norm_to_actual.setdefault(normalise_name(column), column)


def auto_detect(columns):
    detected = {}
    for field, aliases in ALIASES.items():
        detected[field] = None
        for alias in aliases:
            key = normalise_name(alias)
            if key in norm_to_actual:
                detected[field] = norm_to_actual[key]
                break
    return detected


COLUMN_MAP = auto_detect(raw_df.columns)

# ---- Manual overrides -------------------------------------------------------
# If a field is wrong or NOT FOUND, uncomment the line and type the exact
# column name shown in the detected-column list above. Then re-run this cell.
#
# COLUMN_MAP["barangay_id"] = "your_column_name"
# COLUMN_MAP["barangay_name"] = "your_column_name"
# COLUMN_MAP["red_count"] = "your_column_name"
# COLUMN_MAP["yellow_count"] = "your_column_name"
# COLUMN_MAP["black_count"] = "your_column_name"
# COLUMN_MAP["sector_score"] = "your_column_name"
# COLUMN_MAP["avg_lat"] = "your_column_name"
# COLUMN_MAP["avg_lng"] = "your_column_name"
# COLUMN_MAP["bplo_lat"] = "your_column_name"
# COLUMN_MAP["bplo_lng"] = "your_column_name"
# COLUMN_MAP["production_ops"] = "your_column_name"
# COLUMN_MAP["production_rank"] = "your_column_name"
# COLUMN_MAP["dispatchable_red_count"] = "your_column_name"
# COLUMN_MAP["dispatchable_yellow_count"] = "your_column_name"
# COLUMN_MAP["dispatchable_business_count"] = "your_column_name"
# COLUMN_MAP["open_inspection_excluded"] = "your_column_name"

# ---- Fallback settings ------------------------------------------------------
# Use these ONLY when the value is not present as a column.
# BPLO coordinates must come from the production WLC configuration.
BPLO_LAT = None
BPLO_LNG = None

# Set True ONLY if you have confirmed that no barangay has a sector override.
# Then every barangay gets the production neutral score of 50.
SECTOR_OVERRIDES_ABSENT = False

# Set True ONLY if you have confirmed that a blank sector_score means
# "no override exists" and should use the neutral score of 50.
CONFIRM_BLANK_SECTOR_MEANS_NO_OVERRIDE = False

mapping_table = []
for field in ALIASES:
    if field in REQUIRED_FIELDS:
        status = "Required column"
    elif field == "sector_score":
        status = "Required unless SECTOR_OVERRIDES_ABSENT=True"
    elif field in ("bplo_lat", "bplo_lng"):
        status = "Required unless BPLO_LAT/BPLO_LNG are set"
    else:
        status = "Optional"
    mapping_table.append(
        {
            "field": field,
            "status": status,
            "mapped_column": COLUMN_MAP.get(field) if COLUMN_MAP.get(field) else "— NOT FOUND —",
        }
    )

print("Column mapping:")
display(pd.DataFrame(mapping_table))

used_columns = {}
for field, column in COLUMN_MAP.items():
    if column:
        used_columns.setdefault(column, []).append(field)

duplicates = {
    column: fields for column, fields in used_columns.items() if len(fields) > 1
}
if duplicates:
    print("\nWARNING: the same source column is mapped to more than one field:")
    for column, fields in duplicates.items():
        print(f"  - {column!r}: {fields}")
    print("Fix COLUMN_MAP so each field uses the correct column.")

missing_required = [field for field in REQUIRED_FIELDS if not COLUMN_MAP.get(field)]
if missing_required:
    print("\nMissing required columns (map them before continuing):")
    for field in missing_required:
        print(f"  - {field}")
else:
    print("\nAll required columns are mapped. Continue to the validation cell.")

## Validate the data

The next two cells check the mapped data before any score is calculated. They report:

- missing or duplicated mappings
- duplicate barangay IDs or names
- non-numeric, negative, or fractional counts
- a `flagged_count` that does not equal Red + Yellow + Black
- missing or out-of-range sector scores
- missing or invalid mean coordinates
- missing or inconsistent BPLO coordinates
- production OPS/rank problems
- incomplete dispatch-eligibility columns
- zero Yellow or Black records

If anything is an **error**, the notebook stops with a clear message. Warnings do not stop the analysis, but they are shown prominently.

In [ ]:
# ============================================================
# STEP 4a — Validation helpers
# ============================================================
# These helpers only check data. They do not calculate OPS yet.

VALIDATION_ROWS = []
VALIDATION_ERRORS = []
VALIDATION_WARNINGS = []
LIMITATIONS = []
LIMITATION_HTML = []


def add_check(name, status, detail):
    VALIDATION_ROWS.append({"check": name, "status": status, "detail": detail})
    if status == "ERROR":
        VALIDATION_ERRORS.append(f"{name}: {detail}")
    elif status == "WARN":
        VALIDATION_WARNINGS.append(f"{name}: {detail}")


def clean_id(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    text = str(value).strip()
    if re.fullmatch(r"-?\d+\.0+", text):
        return text.split(".")[0]
    return text


def show_rows(mask, columns, limit=20):
    rows = raw_df.loc[mask, columns].head(limit)
    if len(rows) == 0:
        return "no rows"
    return rows.to_string(index=True)


def parse_boolean_series(series):
    def one(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        text = str(value).strip().lower()
        if text in {"true", "1", "yes", "y", "t"}:
            return True
        if text in {"false", "0", "no", "n", "f"}:
            return False
        return np.nan

    return series.map(one)


def round1(values):
    """Python round(..., 1) for every value, matching the production backend."""
    return np.array([round(float(value), 1) for value in values], dtype=float)


def haversine_km_vector(lat1, lng1, lat2, lng2):
    """Haversine distance in km, vectorised. Earth radius 6371.0088 km."""
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lng1 = np.radians(np.asarray(lng1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lng2 = np.radians(np.asarray(lng2, dtype=float))
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2.0) ** 2
    )
    a = np.clip(a, 0.0, 1.0)
    return 2.0 * 6371.0088 * np.arcsin(np.sqrt(a))


print("Validation helpers loaded.")

In [ ]:
# ============================================================
# STEP 4b — Check mappings, IDs, counts, and sector scores
# ============================================================
VALIDATION_ROWS = []
VALIDATION_ERRORS = []
VALIDATION_WARNINGS = []
LIMITATIONS = []
LIMITATION_HTML = []

# ---- 1. Required mapping ----------------------------------------------------
missing_required = [field for field in REQUIRED_FIELDS if not COLUMN_MAP.get(field)]
if missing_required:
    add_check("Required columns mapped", "ERROR", "Missing: " + ", ".join(missing_required))
else:
    add_check("Required columns mapped", "PASS", ", ".join(REQUIRED_FIELDS))

used_columns = {}
for field, column in COLUMN_MAP.items():
    if column:
        used_columns.setdefault(column, []).append(field)
duplicates = {
    column: fields for column, fields in used_columns.items() if len(fields) > 1
}
if duplicates:
    add_check("No source column mapped twice", "ERROR", str(duplicates))
else:
    add_check("No source column mapped twice", "PASS", "Each mapped field has its own column.")

mapped_missing = [column for column in used_columns if column not in raw_df.columns]
if mapped_missing:
    add_check("Mapped columns exist in the file", "ERROR", "Not found: " + ", ".join(mapped_missing))
else:
    add_check("Mapped columns exist in the file", "PASS", "All mapped columns were found.")

if VALIDATION_ERRORS:
    display(pd.DataFrame(VALIDATION_ROWS))
    print("\nSTOP — the column mapping is incomplete or invalid.")
    print("Fix COLUMN_MAP in the previous cell, re-run it, then re-run this cell.")
    raise ValueError("Column mapping is incomplete or invalid. See the messages above.")

# ---- 2. Build canonical columns --------------------------------------------
work = pd.DataFrame(index=raw_df.index)
work["barangay_id"] = raw_df[COLUMN_MAP["barangay_id"]].map(clean_id)
work["barangay_name"] = raw_df[COLUMN_MAP["barangay_name"]].astype(str).str.strip()

for field in ("red_count", "yellow_count", "black_count"):
    source = raw_df[COLUMN_MAP[field]]
    values = pd.to_numeric(source, errors="coerce")
    invalid = values.isna() | (values < 0) | ((values % 1).abs() > 1e-9)
    if invalid.any():
        add_check(
            f"{field} valid",
            "ERROR",
            f"{int(invalid.sum())} invalid value(s). Examples: "
            + show_rows(invalid, [COLUMN_MAP[field]]),
        )
    else:
        add_check(f"{field} valid", "PASS", "Non-negative whole numbers.")
    work[field] = values

# ---- 3. Barangay identifiers ------------------------------------------------
blank_ids = work["barangay_id"].eq("")
if blank_ids.any():
    add_check("Barangay IDs present", "ERROR", f"{int(blank_ids.sum())} blank ID(s).")
else:
    add_check("Barangay IDs present", "PASS", f"{len(work)} row(s).")

blank_names = work["barangay_name"].eq("") | work["barangay_name"].str.lower().eq("nan")
if blank_names.any():
    add_check("Barangay names present", "ERROR", f"{int(blank_names.sum())} blank name(s).")
else:
    add_check("Barangay names present", "PASS", f"{len(work)} row(s).")

duplicate_ids = work["barangay_id"].duplicated(keep=False) & work["barangay_id"].ne("")
if duplicate_ids.any():
    add_check(
        "Barangay IDs unique",
        "ERROR",
        f"Duplicate IDs: {sorted(work.loc[duplicate_ids, 'barangay_id'].unique().tolist())}",
    )
else:
    add_check("Barangay IDs unique", "PASS", "One row per barangay ID.")

duplicate_names = work["barangay_name"].duplicated(keep=False)
if duplicate_names.any():
    add_check(
        "Barangay names unique",
        "WARN",
        f"Duplicate names: {sorted(work.loc[duplicate_names, 'barangay_name'].unique().tolist())}. "
        "This is allowed; ranking uses name and ID as tie-breakers.",
    )
else:
    add_check("Barangay names unique", "PASS", "No duplicate names.")

duplicate_rows = work.duplicated(keep=False)
if duplicate_rows.any():
    add_check(
        "No duplicate rows",
        "WARN",
        f"{int(duplicate_rows.sum())} row(s) are exact duplicates of another row.",
    )
else:
    add_check("No duplicate rows", "PASS", "No exact duplicate rows.")

# ---- 4. Count consistency ---------------------------------------------------
work["flagged_count"] = work[["red_count", "yellow_count", "black_count"]].sum(
    axis=1, min_count=3
)

if COLUMN_MAP.get("flagged_count"):
    supplied = pd.to_numeric(raw_df[COLUMN_MAP["flagged_count"]], errors="coerce")
    invalid_flag = supplied.isna() | (supplied < 0) | ((supplied % 1).abs() > 1e-9)
    if invalid_flag.any():
        add_check(
            "flagged_count valid",
            "ERROR",
            f"{int(invalid_flag.sum())} invalid value(s).",
        )
    mismatch = supplied.ne(work["flagged_count"]) & ~invalid_flag
    if mismatch.any():
        add_check(
            "flagged_count matches Red+Yellow+Black",
            "ERROR",
            f"{int(mismatch.sum())} mismatch(es). Examples: "
            + show_rows(mismatch, [COLUMN_MAP["flagged_count"]]),
        )
    else:
        add_check(
            "flagged_count matches Red+Yellow+Black",
            "PASS",
            "All supplied totals match.",
        )
else:
    add_check(
        "flagged_count matches Red+Yellow+Black",
        "PASS",
        "No flagged_count column supplied; the notebook uses Red+Yellow+Black.",
    )

# ---- 5. Sector scores -------------------------------------------------------
if COLUMN_MAP.get("sector_score"):
    sector_values = pd.to_numeric(raw_df[COLUMN_MAP["sector_score"]], errors="coerce")
    invalid_sector = sector_values.notna() & (
        (sector_values < 0) | (sector_values > 100)
    )
    blank_sector = sector_values.isna()

    if invalid_sector.any():
        add_check(
            "sector_score in 0–100",
            "ERROR",
            f"{int(invalid_sector.sum())} value(s) outside 0–100. Examples: "
            + show_rows(invalid_sector, [COLUMN_MAP["sector_score"]]),
        )
    elif blank_sector.any():
        if CONFIRM_BLANK_SECTOR_MEANS_NO_OVERRIDE:
            sector_values = sector_values.fillna(50.0)
            add_check(
                "sector_score blanks use 50",
                "WARN",
                f"{int(blank_sector.sum())} blank value(s) were confirmed as "
                "'no override' and set to 50.",
            )
        else:
            add_check(
                "sector_score blanks use 50",
                "ERROR",
                f"{int(blank_sector.sum())} blank value(s). Set "
                "CONFIRM_BLANK_SECTOR_MEANS_NO_OVERRIDE=True only if blank means "
                "'no override exists'.",
            )
    else:
        add_check("sector_score in 0–100", "PASS", "All sector scores are valid.")

    work["sector_score"] = sector_values
    if COLUMN_MAP.get("sector_score_source"):
        work["sector_score_source"] = (
            raw_df[COLUMN_MAP["sector_score_source"]].astype(str).str.strip()
        )
    else:
        work["sector_score_source"] = "supplied"

    if work["sector_score"].notna().all():
        add_check(
            "Sector input coverage",
            "PASS",
            f"{work['sector_score'].nunique()} distinct sector score(s); "
            f"{int((work['sector_score'] == 50).sum())} barangay(s) at 50.",
        )
else:
    if SECTOR_OVERRIDES_ABSENT:
        work["sector_score"] = 50.0
        work["sector_score_source"] = "neutral_fallback_confirmed"
        add_check(
            "Sector scores available",
            "WARN",
            "No sector column supplied. SECTOR_OVERRIDES_ABSENT=True, so every "
            "barangay uses 50.",
        )
    else:
        work["sector_score"] = np.nan
        work["sector_score_source"] = "missing"
        add_check(
            "Sector scores available",
            "ERROR",
            "No sector_score column. Supply the configured score, or set "
            "SECTOR_OVERRIDES_ABSENT=True only if you confirmed no override exists.",
        )

print("Count, ID, and sector checks complete.")

In [ ]:
# ============================================================
# STEP 4c — Check coordinates, BPLO, parity inputs, and eligibility
# ============================================================

# ---- 6. Mean coordinates ----------------------------------------------------
for field in ("avg_lat", "avg_lng"):
    source = raw_df[COLUMN_MAP[field]]
    values = pd.to_numeric(source, errors="coerce")
    nonblank = source.notna() & source.astype(str).str.strip().ne("")
    nonnumeric = nonblank & values.isna()

    if field == "avg_lat":
        out_of_range = values.notna() & ((values < -90) | (values > 90))
        valid_range = (-90, 90)
    else:
        out_of_range = values.notna() & ((values < -180) | (values > 180))
        valid_range = (-180, 180)

    if nonnumeric.any():
        add_check(
            f"{field} numeric",
            "WARN",
            f"{int(nonnumeric.sum())} non-numeric value(s) treated as unavailable "
            "(D=50). Examples: " + show_rows(nonnumeric, [COLUMN_MAP[field]]),
        )
    else:
        add_check(f"{field} numeric", "PASS", "All nonblank values are numeric.")

    if out_of_range.any():
        add_check(
            f"{field} in range {valid_range}",
            "WARN",
            f"{int(out_of_range.sum())} out-of-range value(s) treated as unavailable (D=50).",
        )
    else:
        add_check(f"{field} in range {valid_range}", "PASS", "All nonblank values are in range.")

    work[field] = values.where(~out_of_range, np.nan)

coordinate_missing = work["avg_lat"].isna() | work["avg_lng"].isna()
if coordinate_missing.any():
    add_check(
        "Mean coordinates available",
        "WARN",
        f"{int(coordinate_missing.sum())} barangay(s) have no valid mean coordinate "
        "pair; their distance penalty D will be 50 and proximity 50, as the production "
        "rule requires.",
    )
else:
    add_check(
        "Mean coordinates available",
        "PASS",
        "Every barangay has a valid mean coordinate pair.",
    )

# ---- 7. BPLO coordinates ----------------------------------------------------
bplo_lat_column = COLUMN_MAP.get("bplo_lat")
bplo_lng_column = COLUMN_MAP.get("bplo_lng")

if bplo_lat_column and bplo_lng_column:
    bplo_lat_values = pd.to_numeric(raw_df[bplo_lat_column], errors="coerce")
    bplo_lng_values = pd.to_numeric(raw_df[bplo_lng_column], errors="coerce")

    if bplo_lat_values.isna().any() or bplo_lng_values.isna().any():
        add_check(
            "BPLO coordinates valid",
            "ERROR",
            "One or more BPLO latitude/longitude values are blank or non-numeric.",
        )
    elif bplo_lat_values.nunique() != 1 or bplo_lng_values.nunique() != 1:
        add_check(
            "BPLO coordinates constant",
            "ERROR",
            "BPLO latitude/longitude must be the same for every row.",
        )
    else:
        bplo_lat = float(bplo_lat_values.iloc[0])
        bplo_lng = float(bplo_lng_values.iloc[0])
        add_check(
            "BPLO coordinates valid",
            "PASS",
            f"BPLO = ({bplo_lat}, {bplo_lng}).",
        )
elif bplo_lat_column or bplo_lng_column:
    add_check(
        "BPLO coordinates supplied",
        "ERROR",
        "Supply both bplo_lat and bplo_lng columns, or both BPLO_LAT and BPLO_LNG values.",
    )
elif BPLO_LAT is not None and BPLO_LNG is not None:
    bplo_lat = float(BPLO_LAT)
    bplo_lng = float(BPLO_LNG)
    add_check(
        "BPLO coordinates valid",
        "PASS",
        f"Using BPLO_LAT/BPLO_LNG = ({bplo_lat}, {bplo_lng}).",
    )
else:
    bplo_lat = np.nan
    bplo_lng = np.nan
    add_check(
        "BPLO coordinates supplied",
        "ERROR",
        "No BPLO coordinate columns and no BPLO_LAT/BPLO_LNG values. Enter the "
        "production BPLO coordinates in the mapping cell.",
    )

if np.isfinite(bplo_lat) and not (-90 <= bplo_lat <= 90):
    add_check("BPLO latitude in range", "ERROR", f"BPLO latitude {bplo_lat} is outside -90..90.")
if np.isfinite(bplo_lng) and not (-180 <= bplo_lng <= 180):
    add_check("BPLO longitude in range", "ERROR", f"BPLO longitude {bplo_lng} is outside -180..180.")

if (
    bplo_lat_column
    and bplo_lng_column
    and BPLO_LAT is not None
    and BPLO_LNG is not None
    and np.isfinite(bplo_lat)
    and np.isfinite(bplo_lng)
):
    if abs(bplo_lat - float(BPLO_LAT)) > 1e-6 or abs(bplo_lng - float(BPLO_LNG)) > 1e-6:
        add_check(
            "BPLO columns match fallback values",
            "ERROR",
            "The bplo_lat/bplo_lng columns disagree with BPLO_LAT/BPLO_LNG.",
        )

# ---- 8. Optional production OPS and rank -----------------------------------
production_ops_column = COLUMN_MAP.get("production_ops")
production_rank_column = COLUMN_MAP.get("production_rank")
PARITY_COLUMNS_PRESENT = bool(production_ops_column and production_rank_column)

if bool(production_ops_column) != bool(production_rank_column):
    add_check(
        "Production OPS and rank supplied together",
        "ERROR",
        "Supply both production_ops and production_rank, or neither. One alone "
        "cannot support the required parity check.",
    )
elif production_ops_column and production_rank_column:
    production_ops_values = pd.to_numeric(
        raw_df[production_ops_column], errors="coerce"
    )
    production_rank_values = pd.to_numeric(
        raw_df[production_rank_column], errors="coerce"
    )

    if production_ops_values.isna().any():
        add_check(
            "production_ops valid",
            "ERROR",
            f"{int(production_ops_values.isna().sum())} blank/non-numeric OPS value(s).",
        )
    elif ((production_ops_values < 0) | (production_ops_values > 100)).any():
        add_check(
            "production_ops in 0–100",
            "ERROR",
            "One or more production OPS values are outside 0–100.",
        )
    else:
        add_check("production_ops valid", "PASS", "All values are numeric and in 0–100.")

    if production_rank_values.isna().any():
        add_check(
            "production_rank valid",
            "ERROR",
            f"{int(production_rank_values.isna().sum())} blank/non-numeric rank value(s).",
        )
    elif not ((production_rank_values % 1).abs() < 1e-9).all():
        add_check("production_rank valid", "ERROR", "Ranks must be whole numbers.")
    elif production_rank_values.duplicated().any():
        add_check(
            "production_rank unique",
            "ERROR",
            "Production ranks contain duplicates. The full cohort must have one "
            "unique rank per barangay.",
        )
    elif set(production_rank_values.astype(int)) != set(range(1, len(work) + 1)):
        add_check(
            "production_rank covers 1..N",
            "ERROR",
            "Production ranks must cover every integer from 1 to the number of barangays.",
        )
    else:
        add_check("production_rank valid", "PASS", "Unique ranks covering 1..N.")

    work["production_ops"] = production_ops_values
    work["production_rank"] = production_rank_values
    add_check(
        "Production parity inputs",
        "PASS",
        "Both production_ops and production_rank are present; parity will be checked.",
    )
else:
    add_check(
        "Production parity inputs",
        "PASS",
        "production_ops and production_rank are not both supplied; parity will be "
        "reported as NOT CHECKED, not as passed.",
    )

# ---- 9. Optional dispatch eligibility columns ------------------------------
DISPATCH_FIELDS = [
    "dispatchable_red_count",
    "dispatchable_yellow_count",
    "dispatchable_business_count",
    "open_inspection_excluded",
]
dispatch_supplied = [field for field in DISPATCH_FIELDS if COLUMN_MAP.get(field)]
DISPATCH_AVAILABLE = False

if len(dispatch_supplied) == 0:
    add_check(
        "Dispatch eligibility columns",
        "PASS",
        "Not supplied; the dispatch-eligible top-three comparison will be skipped.",
    )
elif len(dispatch_supplied) < len(DISPATCH_FIELDS):
    missing_dispatch = [field for field in DISPATCH_FIELDS if field not in dispatch_supplied]
    add_check(
        "Dispatch eligibility columns",
        "WARN",
        "Incomplete; the dispatch-eligible comparison will be skipped. Missing: "
        + ", ".join(missing_dispatch),
    )
else:
    dispatch_red = pd.to_numeric(
        raw_df[COLUMN_MAP["dispatchable_red_count"]], errors="coerce"
    )
    dispatch_yellow = pd.to_numeric(
        raw_df[COLUMN_MAP["dispatchable_yellow_count"]], errors="coerce"
    )
    dispatch_business = pd.to_numeric(
        raw_df[COLUMN_MAP["dispatchable_business_count"]], errors="coerce"
    )
    open_excluded = parse_boolean_series(
        raw_df[COLUMN_MAP["open_inspection_excluded"]]
    )

    invalid_dispatch = (
        dispatch_red.isna()
        | dispatch_yellow.isna()
        | dispatch_business.isna()
        | (dispatch_red < 0)
        | (dispatch_yellow < 0)
        | (dispatch_business < 0)
        | ((dispatch_red % 1).abs() > 1e-9)
        | ((dispatch_yellow % 1).abs() > 1e-9)
        | ((dispatch_business % 1).abs() > 1e-9)
    )
    invalid_open = open_excluded.isna()

    if invalid_dispatch.any():
        add_check(
            "Dispatch counts valid",
            "ERROR",
            f"{int(invalid_dispatch.sum())} row(s) have blank, negative, or fractional "
            "dispatch counts.",
        )
    else:
        add_check(
            "Dispatch counts valid",
            "PASS",
            "Dispatchable Red/Yellow/business counts are non-negative whole numbers.",
        )

    if invalid_open.any():
        add_check(
            "open_inspection_excluded valid",
            "ERROR",
            f"{int(invalid_open.sum())} row(s) have a non-boolean "
            "open_inspection_excluded value.",
        )
    else:
        add_check(
            "open_inspection_excluded valid",
            "PASS",
            "Boolean values parsed.",
        )

    work["dispatchable_red_count"] = dispatch_red
    work["dispatchable_yellow_count"] = dispatch_yellow
    work["dispatchable_business_count"] = dispatch_business
    work["open_inspection_excluded"] = open_excluded

    if not invalid_dispatch.any() and not invalid_open.any():
        work["dispatch_eligible"] = (
            (dispatch_red + dispatch_yellow > 0)
            & (dispatch_business > 0)
            & (~open_excluded)
        )
        DISPATCH_AVAILABLE = True
        add_check(
            "Dispatch eligibility available",
            "PASS",
            f"{int(work['dispatch_eligible'].sum())} of {len(work)} barangay(s) are "
            "dispatch-eligible. Black-only barangays are not eligible.",
        )
    else:
        DISPATCH_AVAILABLE = False

# ---- 10. Totals and data limitations ----------------------------------------
total_red = int(pd.to_numeric(work["red_count"], errors="coerce").fillna(0).sum())
total_yellow = int(pd.to_numeric(work["yellow_count"], errors="coerce").fillna(0).sum())
total_black = int(pd.to_numeric(work["black_count"], errors="coerce").fillna(0).sum())
total_flags = total_red + total_yellow + total_black
zero_flag_count = int((work["flagged_count"].fillna(0) == 0).sum())
DEGENERATE_COHORT = total_flags == 0

YELLOW_LIMITATION = None
BLACK_LIMITATION = None
SECTOR_LIMITATION = None
COORDINATE_LIMITATION = None

if total_yellow == 0:
    text = (
        "No effective Yellow records are present in this snapshot. The analysis "
        "cannot evaluate how Yellow's weight (2 in weighted burden) affects the "
        "rankings. No Yellow records were created or simulated."
    )
    YELLOW_LIMITATION = text
    LIMITATIONS.append(text)
    LIMITATION_HTML.append(text)
    add_check("Yellow records present", "WARN", text)
else:
    add_check("Yellow records present", "PASS", f"{total_yellow:,} effective Yellow record(s).")

if total_black == 0:
    text = (
        "No effective Black records are present in this snapshot. Black keeps its "
        "production weight of 4 in the formula, but its effect on rankings cannot be "
        "evaluated here. No Black records were created or simulated."
    )
    BLACK_LIMITATION = text
    LIMITATIONS.append(text)
    LIMITATION_HTML.append(text)
    add_check("Black records present", "WARN", text)
else:
    add_check("Black records present", "PASS", f"{total_black:,} effective Black record(s).")

sector_available = work["sector_score"].notna()
if sector_available.all() and work["sector_score"].nunique() == 1:
    sector_value = float(work["sector_score"].iloc[0])
    text = (
        f"All barangays use the same sector score ({sector_value:g}). Changing the "
        "sector weight adds the same constant to every flagged barangay, so it cannot "
        "change their relative order in this snapshot. The grid still scans sector "
        "weights, but sector discrimination cannot be evaluated here."
    )
    SECTOR_LIMITATION = text
    LIMITATIONS.append(text)
    LIMITATION_HTML.append(text)
    add_check("Sector score variation", "WARN", text)
else:
    add_check(
        "Sector score variation",
        "PASS",
        f"{work['sector_score'].nunique()} distinct sector score(s).",
    )

valid_coordinate_count = int((~coordinate_missing).sum())
if valid_coordinate_count <= 1:
    text = (
        f"Only {valid_coordinate_count} barangay(s) have valid mean coordinates. "
        "Proximity is 50 for every other barangay, so the proximity weight has little "
        "or no ability to differentiate them in this snapshot."
    )
    COORDINATE_LIMITATION = text
    LIMITATIONS.append(text)
    LIMITATION_HTML.append(text)
    add_check("Coordinate variation", "WARN", text)
else:
    add_check(
        "Coordinate variation",
        "PASS",
        f"{valid_coordinate_count} of {len(work)} barangay(s) have valid mean coordinates.",
    )

if zero_flag_count:
    add_check(
        "Zero-flag barangays",
        "PASS" if zero_flag_count < len(work) else "WARN",
        f"{zero_flag_count} barangay(s) have zero Red/Yellow/Black records; their OPS "
        "will be exactly 0 under every scenario.",
    )
else:
    add_check("Zero-flag barangays", "PASS", "Every barangay has at least one flagged record.")

if DEGENERATE_COHORT:
    add_check(
        "Non-degenerate cohort",
        "WARN",
        "Every barangay has zero flags. Baseline OPS will be 0 for all rows, so the "
        "weight grid cannot produce a meaningful ranking comparison.",
    )

# ---- 11. Display the validation report -------------------------------------
validation_df = pd.DataFrame(VALIDATION_ROWS)
print("Validation report:")
display(validation_df)

if VALIDATION_WARNINGS:
    print("\nWarnings:")
    for warning in VALIDATION_WARNINGS:
        print(f"  - {warning}")

if LIMITATION_HTML:
    for text in LIMITATION_HTML:
        display(
            HTML(
                "<div style='border:3px solid #b91c1c;background:#fef2f2;"
                "padding:14px;margin:12px 0;border-radius:8px;color:#7f1d1d;'>"
                "<b style='font-size:16px;'>DATA LIMITATION — READ BEFORE INTERPRETING</b>"
                f"<br><br>{text}</div>"
            )
        )

if VALIDATION_ERRORS:
    print("\nSTOP — validation found errors that prevent a production-faithful baseline:")
    for error in VALIDATION_ERRORS:
        print(f"  - {error}")
    print(
        "\nFix the export or the mapping, then re-run the validation cells. "
        "The notebook will not substitute assumptions or simulate missing records."
    )
    raise ValueError("Validation failed. See the error list above.")

# ---- 12. Final working table ------------------------------------------------
ops_input = work.copy()
ops_input["barangay_name_casefold"] = ops_input["barangay_name"].astype(str).str.casefold()

id_numeric = pd.to_numeric(ops_input["barangay_id"], errors="coerce")
if id_numeric.notna().all() and ((id_numeric % 1).abs() < 1e-9).all():
    ops_input["_id_sort"] = id_numeric.astype("int64")
else:
    ops_input["_id_sort"] = ops_input["barangay_id"].astype(str)

n_barangays = len(ops_input)

DATA_SUMMARY = {
    "rows": int(n_barangays),
    "total_red": total_red,
    "total_yellow": total_yellow,
    "total_black": total_black,
    "total_flags": total_flags,
    "zero_flag_barangays": zero_flag_count,
    "bplo_lat": float(bplo_lat),
    "bplo_lng": float(bplo_lng),
    "parity_columns_present": bool(PARITY_COLUMNS_PRESENT),
    "dispatch_available": bool(DISPATCH_AVAILABLE),
    "dispatch_eligible_count": int(work["dispatch_eligible"].sum()) if DISPATCH_AVAILABLE else 0,
    "degenerate_cohort": bool(DEGENERATE_COHORT),
    "sector_distinct_values": int(work["sector_score"].nunique()),
    "valid_coordinate_count": valid_coordinate_count,
}

print("\nCohort summary:")
print(f"  Barangays: {n_barangays}")
print(f"  Red records: {total_red:,}")
print(f"  Yellow records: {total_yellow:,}")
print(f"  Black records: {total_black:,}")
print(f"  Zero-flag barangays: {zero_flag_count}")
print(f"  BPLO: ({bplo_lat}, {bplo_lng})")
print(f"  Production parity columns: {'present' if PARITY_COLUMNS_PRESENT else 'not present'}")
print(f"  Dispatch comparison: {'available' if DISPATCH_AVAILABLE else 'unavailable'}")
print("\nValidation complete. Continue to the production baseline.")

## Production baseline

The production baseline uses exactly:

- Risk weight **0.68**
- Sector weight **0.07**
- Proximity weight **0.25**

The formulas, rounding, zero-flag rule, bands, and tie-breaking are implemented exactly as specified. The reference weighted burden **Q_max** is calculated once across the full municipal cohort and is **not** recalculated for each scenario.

In [ ]:
# ============================================================
# STEP 5 — Calculate the production baseline (0.68 / 0.07 / 0.25)
# ============================================================
assert "ops_input" in globals(), "Run the validation cells successfully before this cell."

BASELINE_WEIGHTS = {"risk": 0.68, "sector": 0.07, "proximity": 0.25}
if abs(sum(BASELINE_WEIGHTS.values()) - 1.0) > 1e-12:
    raise ValueError("Baseline weights must sum to 1.")

red = ops_input["red_count"].to_numpy(dtype=float)
yellow = ops_input["yellow_count"].to_numpy(dtype=float)
black = ops_input["black_count"].to_numpy(dtype=float)
sector = ops_input["sector_score"].to_numpy(dtype=float)
avg_lat = ops_input["avg_lat"].to_numpy(dtype=float)
avg_lng = ops_input["avg_lng"].to_numpy(dtype=float)

# Weighted burden: Q_i = 3*Red + 2*Yellow + 4*Black
burden = 3.0 * red + 2.0 * yellow + 4.0 * black
Q_MAX = float(np.max(burden)) if len(burden) else 0.0

# Risk: round(100 * Q_i / Q_max, 1), or 0 when Q_max is 0
if Q_MAX > 0:
    risk = round1(100.0 * burden / Q_MAX)
else:
    risk = np.zeros(len(burden), dtype=float)

# Distance: Haversine from BPLO to mean dispatchable Red/Yellow coordinates
valid_coordinates = np.isfinite(avg_lat) & np.isfinite(avg_lng)
distance_km = np.full(len(burden), np.nan, dtype=float)
if valid_coordinates.any():
    distance_km[valid_coordinates] = haversine_km_vector(
        bplo_lat,
        bplo_lng,
        avg_lat[valid_coordinates],
        avg_lng[valid_coordinates],
    )

# Distance penalty: round(min(100, 100*d/20), 1); 50 when coordinates unavailable
distance_score = np.where(
    valid_coordinates,
    round1(np.minimum(100.0, 100.0 * distance_km / 20.0)),
    50.0,
)
proximity = round1(100.0 - distance_score)

# OPS: 0 when there are no flags; otherwise clamp(round(weighted sum, 1), 0, 100)
flagged = burden > 0
raw_ops = (
    BASELINE_WEIGHTS["risk"] * risk
    + BASELINE_WEIGHTS["sector"] * sector
    + BASELINE_WEIGHTS["proximity"] * proximity
)
ops_score = np.where(flagged, np.clip(round1(raw_ops), 0.0, 100.0), 0.0)


def band_for(values):
    values = np.asarray(values)
    return np.where(
        values >= 60,
        "High Priority",
        np.where(values >= 30, "Standard", "Monitor"),
    )


def order_indices(values):
    return np.lexsort(
        (
            ops_input["_id_sort"].to_numpy(),
            ops_input["barangay_name_casefold"].to_numpy(),
            -burden,
            -np.asarray(values, dtype=float),
        )
    )


def ranks_for(values):
    order = order_indices(values)
    ranks = np.empty(len(order), dtype=int)
    ranks[order] = np.arange(1, len(order) + 1)
    return ranks


priority_band = band_for(ops_score)
baseline_ranks = ranks_for(ops_score)

baseline = ops_input.copy()
baseline["weighted_burden"] = burden.astype(int)
baseline["risk_score"] = risk
baseline["distance_km"] = np.where(valid_coordinates, np.round(distance_km, 3), np.nan)
baseline["distance_score"] = distance_score
baseline["proximity_score"] = proximity
baseline["ops_score"] = ops_score
baseline["priority_band"] = priority_band
baseline["rank"] = baseline_ranks
baseline["has_flags"] = flagged
baseline["location_missing"] = ~valid_coordinates

# Display-only contributions. They may not add to OPS exactly because OPS is rounded.
baseline["risk_contribution"] = np.round(BASELINE_WEIGHTS["risk"] * risk, 1)
baseline["sector_contribution"] = np.round(BASELINE_WEIGHTS["sector"] * sector, 1)
baseline["proximity_contribution"] = np.round(
    BASELINE_WEIGHTS["proximity"] * proximity, 1
)

baseline = baseline.sort_values("rank").reset_index(drop=True)

print("Baseline weights:", BASELINE_WEIGHTS)
print(f"Q_max (maximum weighted burden across the full cohort): {Q_MAX:g}")
print(f"Barangays: {len(baseline)}")
print(
    "Priority bands: High Priority >= 60; Standard 30 to < 60; Monitor < 30."
)
print(
    "Reminder: OPS is a relative inspection-priority score, not a violation "
    "probability."
)

baseline_columns = [
    "rank",
    "barangay_id",
    "barangay_name",
    "red_count",
    "yellow_count",
    "black_count",
    "weighted_burden",
    "risk_score",
    "sector_score",
    "distance_km",
    "distance_score",
    "proximity_score",
    "ops_score",
    "priority_band",
]
display(baseline[baseline_columns])

In [ ]:
# ============================================================
# Baseline chart
# ============================================================
band_colors = {
    "High Priority": "#dc2626",
    "Standard": "#d97706",
    "Monitor": "#16a34a",
}

plot_baseline = baseline.sort_values("rank", ascending=True)
height = max(4.5, 0.42 * len(plot_baseline))
fig, ax = plt.subplots(figsize=(12, height))

y_positions = np.arange(len(plot_baseline))
bar_colors = [band_colors.get(band, "#64748b") for band in plot_baseline["priority_band"]]
ax.barh(y_positions, plot_baseline["ops_score"], color=bar_colors, edgecolor="white")

ax.set_yticks(y_positions)
ax.set_yticklabels(plot_baseline["barangay_name"], fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("OPS (0–100)")
ax.set_ylabel("Barangay, ordered by baseline rank")
ax.set_title("Production Baseline OPS — Full Municipal Cohort", fontsize=13, pad=12)
ax.grid(axis="x", alpha=0.25, linestyle="--")

for y, value in zip(y_positions, plot_baseline["ops_score"]):
    ax.text(min(value + 1.0, 99.0), y, f"{value:.1f}", va="center", fontsize=8)

from matplotlib.patches import Patch

legend_handles = [
    Patch(facecolor=band_colors["High Priority"], label="High Priority (>= 60)"),
    Patch(facecolor=band_colors["Standard"], label="Standard (30 to < 60)"),
    Patch(facecolor=band_colors["Monitor"], label="Monitor (< 30)"),
]
ax.legend(handles=legend_handles, loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

## Optional parity check

If the upload contains **both** `production_ops` and `production_rank`, the notebook compares the calculated baseline with production for **every barangay**.

- Parity passes only when the maximum absolute OPS difference is **≤ 0.1**.
- If it fails, the notebook **stops** before the sensitivity analysis and explains the mismatch.
- If the columns are absent, parity is reported as **NOT CHECKED** — never as passed.

In [ ]:
# ============================================================
# STEP 6 — Compare the calculated baseline with production OPS/rank
# ============================================================
assert "baseline" in globals(), "Run the baseline cell before this cell."

PARITY_CHECKED = False
PARITY_PASSED = None
PARITY_MAX_ABS_DIFF = None
parity_table = None
PARITY_TOLERANCE = 0.1

if PARITY_COLUMNS_PRESENT:
    # baseline is sorted by rank and already carries the production columns
    # copied from ops_input. Keep them aligned by using those columns directly.
    parity_table = baseline.copy()
    parity_table["production_ops"] = pd.to_numeric(
        parity_table["production_ops"], errors="coerce"
    )
    parity_table["production_rank"] = pd.to_numeric(
        parity_table["production_rank"], errors="coerce"
    ).astype(int)

    raw_ops_difference = (
        parity_table["production_ops"] - parity_table["ops_score"]
    )
    parity_table["ops_difference"] = np.round(raw_ops_difference, 2)
    # Compare the unrounded difference; rounding is for display only.
    parity_table["abs_ops_difference"] = np.abs(raw_ops_difference)
    parity_table["rank_difference"] = (
        parity_table["production_rank"] - parity_table["rank"]
    )
    parity_table["abs_rank_difference"] = parity_table["rank_difference"].abs()

    PARITY_CHECKED = True
    PARITY_MAX_ABS_DIFF = float(parity_table["abs_ops_difference"].max())
    PARITY_PASSED = PARITY_MAX_ABS_DIFF <= PARITY_TOLERANCE + 1e-9

    parity_columns = [
        "barangay_id",
        "barangay_name",
        "production_ops",
        "ops_score",
        "ops_difference",
        "abs_ops_difference",
        "production_rank",
        "rank",
        "rank_difference",
    ]
    print("Row-level parity comparison (production minus notebook baseline):")
    display(parity_table[parity_columns])

    print(f"\nMaximum absolute OPS difference: {PARITY_MAX_ABS_DIFF:.3f}")
    print(f"Parity tolerance: {PARITY_TOLERANCE} (inclusive)")

    rank_mismatches = int((parity_table["rank_difference"] != 0).sum())
    if rank_mismatches:
        print(
            f"\nWARNING: {rank_mismatches} barangay rank(s) differ even though the "
            "OPS comparison is shown above. Check the production rank convention and "
            "tie-breaking. The pass/fail rule is based on the OPS difference."
        )
        display(
            parity_table.loc[
                parity_table["rank_difference"] != 0,
                [
                    "barangay_id",
                    "barangay_name",
                    "production_rank",
                    "rank",
                    "rank_difference",
                ],
            ]
        )
    else:
        print("Ranks also match exactly.")

    if not PARITY_PASSED:
        above_tolerance = parity_table.loc[
            parity_table["abs_ops_difference"] > PARITY_TOLERANCE + 1e-9,
            parity_columns,
        ]
        print("\nBarangays above the 0.1 OPS tolerance:")
        display(above_tolerance)
        print(
            "\nSTOP — baseline parity failed. The sensitivity analysis was not run. "
            "Possible causes include different baseline weights, different effective "
            "colours or eligibility, a different cohort, a different BPLO location, "
            "a different sector score or fallback, a different mean coordinate, or a "
            "different rounding/tie-break convention. Reconcile the export before "
            "interpreting any scenario result."
        )
        raise ValueError(
            "Baseline parity failed: maximum absolute OPS difference "
            f"{PARITY_MAX_ABS_DIFF:.3f} exceeds {PARITY_TOLERANCE}."
        )

    print("\nPARITY PASSED: the maximum absolute OPS difference is within 0.1.")
else:
    print(
        "PARITY NOT CHECKED: production_ops and production_rank were not both supplied. "
        "This is not a pass. Add both columns to verify that the notebook reproduces "
        "production before interpreting the sensitivity results."
    )

SENSITIVITY_ALLOWED = (not PARITY_CHECKED) or bool(PARITY_PASSED)

## Weight grid

The feasible grid uses weights from **0.00 to 1.00 in 0.05 increments**, all nonnegative, and each combination must sum to 1. There are exactly **231** such combinations.

The exact baseline **(0.68, 0.07, 0.25)** is added as a separate scenario because 0.68 and 0.07 are not multiples of 0.05. Total scenarios: **232**.

In [ ]:
# ============================================================
# STEP 7 — Build the complete feasible weight grid
# ============================================================
assert "baseline" in globals(), "Run the baseline cell before this cell."
if not SENSITIVITY_ALLOWED:
    raise ValueError("Sensitivity analysis is not allowed because baseline parity failed.")

grid = []
for risk_units in range(21):
    for sector_units in range(21 - risk_units):
        proximity_units = 20 - risk_units - sector_units
        grid.append(
            {
                "w_risk": risk_units / 20.0,
                "w_sector": sector_units / 20.0,
                "w_proximity": proximity_units / 20.0,
            }
        )

if len(grid) != 231:
    raise AssertionError(f"Expected 231 feasible weight combinations, found {len(grid)}.")

for scenario in grid:
    total = scenario["w_risk"] + scenario["w_sector"] + scenario["w_proximity"]
    if abs(total - 1.0) > 1e-12:
        raise AssertionError(f"Grid weights do not sum to 1: {scenario}")
    if min(scenario.values()) < 0:
        raise AssertionError(f"Grid contains a negative weight: {scenario}")

baseline_spec = {
    "scenario_id": "baseline_AHP",
    "scenario_label": "Exact baseline 0.68 / 0.07 / 0.25",
    "w_risk": 0.68,
    "w_sector": 0.07,
    "w_proximity": 0.25,
    "is_baseline": True,
}

grid_specs = []
for scenario in grid:
    risk_percent = int(round(scenario["w_risk"] * 100))
    sector_percent = int(round(scenario["w_sector"] * 100))
    proximity_percent = int(round(scenario["w_proximity"] * 100))
    grid_specs.append(
        {
            "scenario_id": (
                f"grid_r{risk_percent:03d}_s{sector_percent:03d}_p{proximity_percent:03d}"
            ),
            "scenario_label": (
                f"Grid {scenario['w_risk']:.2f} / {scenario['w_sector']:.2f} / "
                f"{scenario['w_proximity']:.2f}"
            ),
            "w_risk": scenario["w_risk"],
            "w_sector": scenario["w_sector"],
            "w_proximity": scenario["w_proximity"],
            "is_baseline": False,
        }
    )

scenario_specs = [baseline_spec] + grid_specs
scenario_ids = [spec["scenario_id"] for spec in scenario_specs]

if len(scenario_ids) != len(set(scenario_ids)):
    raise AssertionError("Scenario IDs are not unique.")

N_GRID = len(grid_specs)
N_SCENARIOS = len(scenario_specs)

print(f"Feasible grid scenarios: {N_GRID}")
print(f"Exact baseline scenarios: 1")
print(f"Total scenarios: {N_SCENARIOS}")
print("\nFirst five grid combinations:")
display(pd.DataFrame(grid_specs[:5])[["scenario_id", "w_risk", "w_sector", "w_proximity"]])
print("Last five grid combinations:")
display(pd.DataFrame(grid_specs[-5:])[["scenario_id", "w_risk", "w_sector", "w_proximity"]])
print("Exact baseline scenario:")
display(pd.DataFrame([baseline_spec])[["scenario_id", "w_risk", "w_sector", "w_proximity"]])

In [ ]:
# ============================================================
# STEP 8 — Recalculate OPS and ranks for every scenario
# ============================================================
assert "scenario_specs" in globals(), "Run the weight-grid cell before this cell."
if DEGENERATE_COHORT:
    raise ValueError(
        "The cohort has zero effective Red, Yellow, and Black records in every "
        "barangay. Every OPS is 0, so a weight-sensitivity ranking comparison is not "
        "meaningful. No scenarios were run."
    )


def score_weights(w_risk, w_sector, w_proximity):
    raw = w_risk * risk + w_sector * sector + w_proximity * proximity
    return np.where(flagged, np.clip(round1(raw), 0.0, 100.0), 0.0)


scenario_ops = np.empty((N_SCENARIOS, n_barangays), dtype=float)
scenario_ranks = np.empty((N_SCENARIOS, n_barangays), dtype=int)

for index, spec in enumerate(scenario_specs):
    values = score_weights(spec["w_risk"], spec["w_sector"], spec["w_proximity"])
    scenario_ops[index] = values
    scenario_ranks[index] = ranks_for(values)
    if index % 50 == 0 or index == N_SCENARIOS - 1:
        print(f"Scored {index + 1}/{N_SCENARIOS}: {spec['scenario_id']}")

scenario_bands = band_for(scenario_ops)

# The first scenario is the exact baseline; verify that it reproduces the baseline.
if not np.allclose(scenario_ops[0], ops_score, atol=1e-12):
    raise AssertionError("The exact baseline scenario does not reproduce the baseline OPS.")
if not np.array_equal(scenario_ranks[0], baseline_ranks):
    raise AssertionError("The exact baseline scenario does not reproduce the baseline ranks.")
if not np.array_equal(scenario_bands[0], priority_band):
    raise AssertionError("The exact baseline scenario does not reproduce the priority bands.")

scenario_weights = np.array(
    [[spec["w_risk"], spec["w_sector"], spec["w_proximity"]] for spec in scenario_specs]
)

print("\nAll scenarios scored with the same fixed cohort and the same Q_max.")
print("Scenario matrix shape:", scenario_ops.shape)

In [ ]:
# ============================================================
# STEP 9 — Compare every scenario with the exact baseline
# ============================================================
assert "scenario_ops" in globals(), "Run the scenario engine before this cell."


def top_indices(ranks, k):
    k = min(k, len(ranks))
    return set(np.where(ranks <= k)[0].tolist())


def jaccard(set_a, set_b):
    union = set_a | set_b
    if not union:
        return np.nan
    return len(set_a & set_b) / len(union)


def safe_spearman(ranks_a, ranks_b):
    try:
        return float(stats.spearmanr(ranks_a, ranks_b).statistic)
    except AttributeError:
        return float(stats.spearmanr(ranks_a, ranks_b)[0])


def safe_kendall_tau_b(ranks_a, ranks_b):
    try:
        return float(stats.kendalltau(ranks_a, ranks_b, variant="b").statistic)
    except TypeError:
        return float(stats.kendalltau(ranks_a, ranks_b)[0])


baseline_top1 = top_indices(baseline_ranks, 1)
baseline_top3 = top_indices(baseline_ranks, 3)
baseline_top5 = top_indices(baseline_ranks, 5)

if DISPATCH_AVAILABLE:
    dispatch_eligible_mask = ops_input["dispatch_eligible"].to_numpy(dtype=bool)
    dispatch_eligible_count = int(dispatch_eligible_mask.sum())

    def dispatch_top_indices(ranks):
        eligible_indices = np.where(dispatch_eligible_mask)[0]
        if len(eligible_indices) == 0:
            return set()
        ordered = eligible_indices[
            np.argsort(ranks[eligible_indices], kind="stable")
        ]
        return set(ordered[: min(3, len(ordered))].tolist())

    baseline_dispatch_top3 = dispatch_top_indices(baseline_ranks)
else:
    dispatch_eligible_mask = None
    dispatch_eligible_count = 0
    baseline_dispatch_top3 = set()

summary_rows = []

for index, spec in enumerate(scenario_specs):
    ranks = scenario_ranks[index]
    ops = scenario_ops[index]
    bands = scenario_bands[index]

    top1 = top_indices(ranks, 1)
    top3 = top_indices(ranks, 3)
    top5 = top_indices(ranks, 5)

    abs_rank_change = np.abs(ranks - baseline_ranks)
    band_agreement_count = int(np.sum(bands == priority_band))

    row = {
        "scenario_id": spec["scenario_id"],
        "scenario_label": spec["scenario_label"],
        "is_baseline": spec["is_baseline"],
        "w_risk": spec["w_risk"],
        "w_sector": spec["w_sector"],
        "w_proximity": spec["w_proximity"],
        "n_barangays": n_barangays,
        "top1_overlap": len(top1 & baseline_top1),
        "top3_overlap": len(top3 & baseline_top3),
        "top5_overlap": len(top5 & baseline_top5),
        "top1_overlap_fraction": len(top1 & baseline_top1) / 1.0,
        "top3_overlap_fraction": len(top3 & baseline_top3) / min(3, n_barangays),
        "top5_overlap_fraction": len(top5 & baseline_top5) / min(5, n_barangays),
        "top3_jaccard": jaccard(top3, baseline_top3),
        "spearman": safe_spearman(baseline_ranks, ranks),
        "kendall_tau_b": safe_kendall_tau_b(baseline_ranks, ranks),
        "mean_abs_rank_change": float(np.mean(abs_rank_change)),
        "max_abs_rank_change": int(np.max(abs_rank_change)),
        "band_agreement_count": band_agreement_count,
        "band_agreement_pct": band_agreement_count / n_barangays * 100.0,
        "dispatch_eligible_count": dispatch_eligible_count,
    }

    if DISPATCH_AVAILABLE:
        scenario_dispatch_top3 = dispatch_top_indices(ranks)
        row["dispatch_top3_overlap"] = len(
            scenario_dispatch_top3 & baseline_dispatch_top3
        )
        row["dispatch_top3_jaccard"] = jaccard(
            scenario_dispatch_top3, baseline_dispatch_top3
        )
    else:
        row["dispatch_top3_overlap"] = np.nan
        row["dispatch_top3_jaccard"] = np.nan

    summary_rows.append(row)

scenario_summary = pd.DataFrame(summary_rows)
summary_grid = scenario_summary.loc[~scenario_summary["is_baseline"]].copy()

metric_columns = [
    "top1_overlap",
    "top3_overlap",
    "top5_overlap",
    "top3_jaccard",
    "spearman",
    "kendall_tau_b",
    "mean_abs_rank_change",
    "max_abs_rank_change",
    "band_agreement_pct",
]
if DISPATCH_AVAILABLE:
    metric_columns += ["dispatch_top3_overlap", "dispatch_top3_jaccard"]

print("Scenario summary (first 5 rows):")
display(scenario_summary.head(5))
print("Scenario summary (last 5 rows):")
display(scenario_summary.tail(5))

print("\nGrid-only descriptive statistics (no pass/fail threshold is applied):")
display(summary_grid[metric_columns].describe().T)

compact_columns = [
    "scenario_id",
    "w_risk",
    "w_sector",
    "w_proximity",
    "top1_overlap",
    "top3_overlap",
    "top5_overlap",
    "top3_jaccard",
    "spearman",
    "kendall_tau_b",
    "mean_abs_rank_change",
    "max_abs_rank_change",
    "band_agreement_pct",
]
print("\nCompact scenario summary (first 10 grid scenarios):")
display(scenario_summary.loc[1:10, compact_columns])
print("The full 232-scenario table is in the CSV and Excel export.")

In [ ]:
# ============================================================
# STEP 10 — Per-barangay rank stability and top-three frequency
# ============================================================
assert "scenario_ranks" in globals(), "Run the scenario summary cell before this cell."


def most_common_band(values):
    counts = pd.Series(values).value_counts()
    maximum = counts.max()
    tied = sorted(counts.loc[counts == maximum].index.tolist())
    return tied[0]


stability = ops_input[
    [
        "barangay_id",
        "barangay_name",
        "red_count",
        "yellow_count",
        "black_count",
        "sector_score",
    ]
].copy()
stability["weighted_burden"] = burden.astype(int)
stability["baseline_rank"] = baseline_ranks
stability["baseline_ops"] = ops_score
stability["baseline_band"] = priority_band
stability["min_rank"] = scenario_ranks.min(axis=0)
stability["max_rank"] = scenario_ranks.max(axis=0)
stability["rank_range"] = stability["max_rank"] - stability["min_rank"]
stability["mean_rank"] = scenario_ranks.mean(axis=0)
stability["rank_std"] = scenario_ranks.std(axis=0)
stability["top1_count"] = (scenario_ranks <= 1).sum(axis=0)
stability["top3_count"] = (scenario_ranks <= 3).sum(axis=0)
stability["top5_count"] = (scenario_ranks <= 5).sum(axis=0)
stability["top3_pct"] = stability["top3_count"] / N_SCENARIOS * 100.0
stability["top3_count_grid"] = (scenario_ranks[1:] <= 3).sum(axis=0)
stability["top3_pct_grid"] = stability["top3_count_grid"] / N_GRID * 100.0
stability["band_agreement_pct"] = (
    (scenario_bands == priority_band).sum(axis=0) / N_SCENARIOS * 100.0
)
stability["band_agreement_pct_grid"] = (
    (scenario_bands[1:] == priority_band).sum(axis=0) / N_GRID * 100.0
)

most_common = []
for column_index in range(n_barangays):
    most_common.append(most_common_band(scenario_bands[:, column_index]))
stability["most_common_band"] = most_common

minimum_scenario = []
maximum_scenario = []
for column_index in range(n_barangays):
    min_index = int(np.argmin(scenario_ranks[:, column_index]))
    max_index = int(np.argmax(scenario_ranks[:, column_index]))
    minimum_scenario.append(scenario_ids[min_index])
    maximum_scenario.append(scenario_ids[max_index])
stability["rank_min_scenario"] = minimum_scenario
stability["rank_max_scenario"] = maximum_scenario

if DISPATCH_AVAILABLE:
    stability["dispatch_eligible"] = ops_input["dispatch_eligible"].to_numpy(dtype=bool)
else:
    stability["dispatch_eligible"] = np.nan

stability = stability.sort_values("baseline_rank").reset_index(drop=True)

stability_columns = [
    "baseline_rank",
    "barangay_id",
    "barangay_name",
    "baseline_ops",
    "baseline_band",
    "min_rank",
    "max_rank",
    "rank_range",
    "mean_rank",
    "top3_count",
    "top3_pct",
    "top3_pct_grid",
    "band_agreement_pct",
    "band_agreement_pct_grid",
    "most_common_band",
]
display(stability[stability_columns])

print(
    "\nRank range, top-three frequency, and band agreement are descriptive. "
    "A wide range or low top-three frequency means the barangay's position depends "
    "on the chosen weights; it is not a pass/fail result."
)

In [ ]:
# ============================================================
# STEP 11 — Distribution summaries and worst-case scenarios
# ============================================================
assert "summary_grid" in globals(), "Run the scenario summary cell before this cell."

metric_labels = {
    "top1_overlap": "Top-1 overlap count",
    "top3_overlap": "Top-3 overlap count",
    "top5_overlap": "Top-5 overlap count",
    "top3_jaccard": "Top-3 Jaccard",
    "spearman": "Spearman rank correlation",
    "kendall_tau_b": "Kendall tau-b",
    "mean_abs_rank_change": "Mean absolute rank change",
    "max_abs_rank_change": "Maximum absolute rank change",
    "band_agreement_pct": "Priority-band agreement (%)",
}
if DISPATCH_AVAILABLE:
    metric_labels["dispatch_top3_overlap"] = "Dispatch-eligible top-3 overlap count"
    metric_labels["dispatch_top3_jaccard"] = "Dispatch-eligible top-3 Jaccard"

summary_stat_rows = []
for column, label in metric_labels.items():
    values = summary_grid[column].dropna()
    if len(values) == 0:
        continue
    summary_stat_rows.append(
        {
            "metric": label,
            "count": int(len(values)),
            "mean": float(values.mean()),
            "median": float(values.median()),
            "std": float(values.std()),
            "min": float(values.min()),
            "max": float(values.max()),
        }
    )

summary_stats = pd.DataFrame(summary_stat_rows)
print("Grid-only distribution summary:")
display(summary_stats)

worst_definitions = [
    ("top3_jaccard", True, "Lowest top-3 Jaccard"),
    ("spearman", True, "Lowest Spearman correlation"),
    ("max_abs_rank_change", False, "Largest maximum rank change"),
    ("band_agreement_pct", True, "Lowest priority-band agreement"),
]
if DISPATCH_AVAILABLE:
    worst_definitions.append(
        ("dispatch_top3_jaccard", True, "Lowest dispatch-eligible top-3 Jaccard")
    )

worst_tables = []
for column, ascending, label in worst_definitions:
    available = summary_grid.dropna(subset=[column])
    if len(available) == 0:
        continue
    table = available.sort_values(
        [column, "scenario_id"], ascending=[ascending, True]
    ).head(5)
    table = table[
        [
            "scenario_id",
            "w_risk",
            "w_sector",
            "w_proximity",
            "top3_jaccard",
            "spearman",
            "kendall_tau_b",
            "max_abs_rank_change",
            "band_agreement_pct",
        ]
    ].copy()
    table.insert(0, "worst_case_metric", label)
    worst_tables.append(table)

worst_cases = (
    pd.concat(worst_tables, ignore_index=True)
    if worst_tables
    else pd.DataFrame()
)

print("\nWorst-case scenarios inside the grid (descriptive only; no pass/fail threshold):")
display(worst_cases)

band_rows = []
for band in ["High Priority", "Standard", "Monitor"]:
    mask = priority_band == band
    if not mask.any():
        continue
    agreement = (
        (scenario_bands[1:, mask] == priority_band[mask]).mean(axis=1) * 100.0
    )
    band_rows.append(
        {
            "baseline_band": band,
            "barangays": int(mask.sum()),
            "mean_agreement_pct": float(agreement.mean()),
            "min_agreement_pct": float(agreement.min()),
            "max_agreement_pct": float(agreement.max()),
        }
    )

band_agreement_by_band = pd.DataFrame(band_rows)
print("\nPriority-band agreement by baseline band:")
display(band_agreement_by_band)

print(
    "\nThese summaries describe what happened inside this weight grid. "
    "They do not establish a stable/unstable threshold, predictive accuracy, or "
    "better inspection outcomes."
)

In [ ]:
# ============================================================
# Charts 1 — distributions across the 231 grid scenarios
# ============================================================
assert "summary_grid" in globals(), "Run the scenario summary cell before this cell."

distribution_panels = [
    ("top3_jaccard", "Top-3 Jaccard overlap with baseline", "Jaccard", (0.0, 1.0)),
    ("spearman", "Spearman rank correlation with baseline", "Spearman rho", (-1.0, 1.0)),
    ("max_abs_rank_change", "Maximum absolute rank change", "Rank positions", None),
    ("band_agreement_pct", "Priority-band agreement with baseline", "Agreement (%)", (0.0, 100.0)),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.ravel()

for axis, (column, title, xlabel, limits) in zip(axes, distribution_panels):
    values = summary_grid[column].dropna()
    bins = min(24, max(8, len(values) // 8))
    axis.hist(values, bins=bins, color="#2563eb", edgecolor="white", alpha=0.85)
    axis.axvline(values.mean(), color="#dc2626", linewidth=2, label=f"Mean = {values.mean():.3f}")
    axis.set_title(title, fontsize=11)
    axis.set_xlabel(xlabel)
    axis.set_ylabel("Number of scenarios")
    if limits is not None:
        axis.set_xlim(*limits)
    axis.grid(axis="y", alpha=0.2, linestyle="--")
    axis.legend(frameon=False, fontsize=9)

fig.suptitle(
    "Distribution Across 231 Feasible Weight Combinations (exact baseline excluded)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

print(
    "The red line is the average of the grid scenarios. It is not a stability "
    "threshold or a pass/fail marker."
)

In [ ]:
# ============================================================
# Charts 2 — heatmaps over the feasible weight triangle
# ============================================================
assert "summary_grid" in globals(), "Run the scenario summary cell before this cell."

weight_axis = np.round(np.arange(21) / 20.0, 2)


def make_weight_matrix(value_column):
    matrix = np.full((21, 21), np.nan, dtype=float)
    for _, row in summary_grid.iterrows():
        risk_index = int(round(row["w_risk"] * 20))
        sector_index = int(round(row["w_sector"] * 20))
        matrix[risk_index, sector_index] = row[value_column]
    return matrix


def draw_weight_heatmap(axis, value_column, title, annotate=False):
    matrix = make_weight_matrix(value_column)
    masked = np.ma.masked_invalid(matrix)
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("white")

    finite_values = matrix[np.isfinite(matrix)]
    vmin = 0.0 if value_column == "top3_jaccard" else float(finite_values.min())
    vmax = 1.0

    image = axis.imshow(
        masked,
        origin="lower",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        interpolation="nearest",
    )

    tick_positions = list(range(0, 21, 2))
    axis.set_xticks(tick_positions)
    axis.set_xticklabels([f"{weight_axis[i]:.1f}" for i in tick_positions], fontsize=8)
    axis.set_yticks(tick_positions)
    axis.set_yticklabels([f"{weight_axis[i]:.1f}" for i in tick_positions], fontsize=8)
    axis.set_xlabel("Sector weight (w2)")
    axis.set_ylabel("Risk weight (w1)")
    axis.set_title(title, fontsize=11)

    if annotate:
        for risk_index in range(21):
            for sector_index in range(21):
                value = matrix[risk_index, sector_index]
                if np.isfinite(value):
                    text_color = "white" if value < 0.55 else "black"
                    axis.text(
                        sector_index,
                        risk_index,
                        f"{value:.2f}",
                        ha="center",
                        va="center",
                        fontsize=5.2,
                        color=text_color,
                    )

    axis.scatter(
        [0.07 * 20],
        [0.68 * 20],
        marker="*",
        s=220,
        color="#ef4444",
        edgecolor="white",
        linewidth=1.2,
        label="Exact baseline 0.68/0.07/0.25",
        zorder=5,
    )
    axis.legend(loc="upper right", fontsize=8, framealpha=0.9)
    axis.set_xlim(-0.5, 20.5)
    axis.set_ylim(-0.5, 20.5)
    return image


fig, axes = plt.subplots(1, 2, figsize=(16, 7.2))

image_top3 = draw_weight_heatmap(
    axes[0],
    "top3_jaccard",
    "Top-3 Jaccard overlap with baseline",
    annotate=True,
)
fig.colorbar(image_top3, ax=axes[0], fraction=0.046, pad=0.04)

image_spearman = draw_weight_heatmap(
    axes[1],
    "spearman",
    "Spearman rank correlation with baseline",
    annotate=False,
)
fig.colorbar(image_spearman, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle(
    "Weight Sensitivity — Proximity weight = 1 - Risk weight - Sector weight; "
    "white cells are infeasible",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Charts 3 — per-barangay rank range and top-three frequency
# ============================================================
assert "stability" in globals(), "Run the stability cell before this cell."

band_colors = globals().get(
    "band_colors",
    {
        "High Priority": "#dc2626",
        "Standard": "#d97706",
        "Monitor": "#16a34a",
    },
)

top_n = min(15, len(stability))
rank_plot = stability.head(top_n).copy()
y_positions = np.arange(len(rank_plot))

fig, axes = plt.subplots(1, 2, figsize=(15, max(5.5, 0.42 * len(rank_plot))))

axis = axes[0]
axis.hlines(
    y_positions,
    rank_plot["min_rank"],
    rank_plot["max_rank"],
    color="#94a3b8",
    linewidth=5,
    alpha=0.65,
    label="Observed rank range",
)
axis.scatter(
    rank_plot["baseline_rank"],
    y_positions,
    color="#dc2626",
    s=45,
    zorder=4,
    label="Baseline rank",
)
axis.scatter(
    rank_plot["mean_rank"],
    y_positions,
    color="#1d4ed8",
    s=30,
    marker="D",
    zorder=4,
    label="Mean rank",
)
axis.set_yticks(y_positions)
axis.set_yticklabels(rank_plot["barangay_name"], fontsize=9)
axis.invert_yaxis()
axis.set_xlabel("Rank (1 = highest priority)")
axis.set_title("Rank range across all scenarios", fontsize=11)
axis.grid(axis="x", alpha=0.25, linestyle="--")
rank_handles, rank_labels = axis.get_legend_handles_labels()

axis = axes[1]
bar_colors = [band_colors.get(band, "#64748b") for band in rank_plot["baseline_band"]]
axis.barh(
    y_positions,
    rank_plot["top3_pct_grid"],
    color=bar_colors,
    edgecolor="white",
    alpha=0.9,
)
for y, value in zip(y_positions, rank_plot["top3_pct_grid"]):
    axis.text(min(value + 1.5, 99.0), y, f"{value:.1f}%", va="center", fontsize=8)
axis.set_yticks(y_positions)
axis.set_yticklabels(rank_plot["barangay_name"], fontsize=9)
axis.invert_yaxis()
axis.set_xlim(0, 100)
axis.set_xlabel("Share of 231 grid scenarios in the top three (%)")
axis.set_title("Top-three frequency across the grid", fontsize=11)
axis.grid(axis="x", alpha=0.25, linestyle="--")

fig.suptitle(
    "Per-Barangay Rank Stability (top 15 by baseline rank; bars use baseline band colours)",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0.06, 1, 0.96))
fig.legend(
    rank_handles,
    rank_labels,
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(0.5, -0.01),
)
plt.show()

In [ ]:
# ============================================================
# STEP 12 — Export CSV and Excel results
# ============================================================
assert "scenario_summary" in globals(), "Run the scenario and stability cells before this cell."

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
INPUT_FILE_NAME = DATA_PATH.name

OUTPUT_DIR = (
    Path("/content/ops_sensitivity_outputs")
    if IN_COLAB
    else Path.cwd() / "ops_sensitivity_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Long-format results: one row per scenario x barangay.
long_frames = []
for scenario_index, spec in enumerate(scenario_specs):
    frame = pd.DataFrame(
        {
            "scenario_id": spec["scenario_id"],
            "scenario_label": spec["scenario_label"],
            "w_risk": spec["w_risk"],
            "w_sector": spec["w_sector"],
            "w_proximity": spec["w_proximity"],
            "is_baseline": spec["is_baseline"],
            "barangay_id": ops_input["barangay_id"].to_numpy(),
            "barangay_name": ops_input["barangay_name"].to_numpy(),
            "red_count": ops_input["red_count"].to_numpy(),
            "yellow_count": ops_input["yellow_count"].to_numpy(),
            "black_count": ops_input["black_count"].to_numpy(),
            "weighted_burden": burden.astype(int),
            "sector_score": ops_input["sector_score"].to_numpy(),
            "ops_score": scenario_ops[scenario_index],
            "rank": scenario_ranks[scenario_index],
            "priority_band": scenario_bands[scenario_index],
            "baseline_rank": baseline_ranks,
            "rank_change_vs_baseline": (
                scenario_ranks[scenario_index] - baseline_ranks
            ),
            "baseline_band": priority_band,
            "band_changed_vs_baseline": (
                scenario_bands[scenario_index] != priority_band
            ),
        }
    )
    long_frames.append(frame)

scenario_rankings_long = pd.concat(long_frames, ignore_index=True)

parity_status = (
    "PASSED"
    if PARITY_CHECKED and PARITY_PASSED
    else ("FAILED" if PARITY_CHECKED else "NOT CHECKED")
)

readme_rows = [
    ("Notebook", "REVELA WLC Operational Priority Score (OPS) — weight sensitivity"),
    ("Run timestamp", RUN_TIMESTAMP),
    ("Input file", INPUT_FILE_NAME),
    ("Barangays in cohort", n_barangays),
    ("Baseline weights", "Risk 0.68, Sector 0.07, Proximity 0.25"),
    ("Weight grid", "231 feasible combinations (0.00-1.00, step 0.05, sum=1) plus 1 exact baseline"),
    ("Q_max", Q_MAX),
    ("Total Red records", total_red),
    ("Total Yellow records", total_yellow),
    ("Total Black records", total_black),
    ("Zero-flag barangays", zero_flag_count),
    ("BPLO latitude", bplo_lat),
    ("BPLO longitude", bplo_lng),
    ("Production OPS/rank parity", parity_status),
    ("Max absolute OPS difference", PARITY_MAX_ABS_DIFF if PARITY_MAX_ABS_DIFF is not None else "n/a"),
    ("Dispatch comparison", "available" if DISPATCH_AVAILABLE else "not available"),
    ("Yellow limitation", YELLOW_LIMITATION if YELLOW_LIMITATION else "none"),
    ("Black limitation", BLACK_LIMITATION if BLACK_LIMITATION else "none"),
    ("Sector limitation", SECTOR_LIMITATION if SECTOR_LIMITATION else "none"),
    ("Coordinate limitation", COORDINATE_LIMITATION if COORDINATE_LIMITATION else "none"),
    (
        "Interpretation",
        "OPS is a relative inspection-priority score, not a violation probability. "
        "This analysis measures ranking robustness under changed weights. It does not "
        "prove predictive accuracy or better inspection outcomes.",
    ),
    (
        "Cohort rule",
        "The full municipal cohort is fixed across all scenarios. No dashboard filters "
        "were applied and Q_max was not recomputed per scenario.",
    ),
]
readme_df = pd.DataFrame(readme_rows, columns=["Item", "Value"])

column_mapping_df = pd.DataFrame(mapping_table)

csv_paths = {}
csv_paths["baseline"] = OUTPUT_DIR / "ops_sensitivity_baseline.csv"
csv_paths["scenario_summary"] = OUTPUT_DIR / "ops_sensitivity_scenario_summary.csv"
csv_paths["stability"] = OUTPUT_DIR / "ops_sensitivity_barangay_stability.csv"
csv_paths["scenario_rankings"] = OUTPUT_DIR / "ops_sensitivity_scenario_rankings.csv"
csv_paths["summary_stats"] = OUTPUT_DIR / "ops_sensitivity_summary_stats.csv"
csv_paths["worst_cases"] = OUTPUT_DIR / "ops_sensitivity_worst_cases.csv"
csv_paths["band_agreement"] = OUTPUT_DIR / "ops_sensitivity_band_agreement.csv"

baseline.to_csv(csv_paths["baseline"], index=False, encoding="utf-8-sig")
scenario_summary.to_csv(csv_paths["scenario_summary"], index=False, encoding="utf-8-sig")
stability.to_csv(csv_paths["stability"], index=False, encoding="utf-8-sig")
scenario_rankings_long.to_csv(
    csv_paths["scenario_rankings"], index=False, encoding="utf-8-sig"
)
summary_stats.to_csv(csv_paths["summary_stats"], index=False, encoding="utf-8-sig")
worst_cases.to_csv(csv_paths["worst_cases"], index=False, encoding="utf-8-sig")
band_agreement_by_band.to_csv(
    csv_paths["band_agreement"], index=False, encoding="utf-8-sig"
)

if PARITY_CHECKED and parity_table is not None:
    csv_paths["parity"] = OUTPUT_DIR / "ops_sensitivity_parity.csv"
    parity_table.to_csv(csv_paths["parity"], index=False, encoding="utf-8-sig")

excel_path = OUTPUT_DIR / "ops_sensitivity_results.xlsx"

excel_sheets = {
    "README": readme_df,
    "Validation": validation_df,
    "Column_Mapping": column_mapping_df,
    "Baseline": baseline,
    "Scenario_Summary": scenario_summary,
    "Summary_Stats": summary_stats,
    "Barangay_Stability": stability,
    "Band_Agreement": band_agreement_by_band,
    "Worst_Cases": worst_cases,
}
if PARITY_CHECKED and parity_table is not None:
    excel_sheets["Parity"] = parity_table
if len(scenario_rankings_long) <= 900_000:
    excel_sheets["Scenario_Rankings"] = scenario_rankings_long
else:
    readme_df = pd.concat(
        [
            readme_df,
            pd.DataFrame(
                [
                    (
                        "Large long-format sheet",
                        "Scenario_Rankings was written to CSV only because it exceeds "
                        "900,000 rows.",
                    )
                ],
                columns=["Item", "Value"],
            ),
        ],
        ignore_index=True,
    )
    excel_sheets["README"] = readme_df

from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in excel_sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        worksheet = writer.sheets[sheet_name[:31]]
        worksheet.freeze_panes = "A2"
        for cell in worksheet[1]:
            cell.font = Font(bold=True)
        if len(frame) <= 5_000:
            for column_index, column_name in enumerate(frame.columns, start=1):
                values = frame[column_name].tolist()
                width = max(
                    [len(str(column_name))]
                    + [len(str(value)) for value in values]
                )
                worksheet.column_dimensions[get_column_letter(column_index)].width = min(
                    max(width + 2, 10), 60
                )

print("Files written to:", OUTPUT_DIR)
for path in [excel_path, *csv_paths.values()]:
    print(f"  - {path.name} ({path.stat().st_size:,} bytes)")

AUTO_DOWNLOAD = True
if AUTO_DOWNLOAD and IN_COLAB:
    from google.colab import files

    download_paths = [excel_path, *csv_paths.values()]
    for path in download_paths:
        files.download(str(path))
else:
    print(
        "\nNot running in Colab, or AUTO_DOWNLOAD=False. Open the files from the "
        "output folder above."
    )

## How to read the sensitivity results

- **Top-1 / top-3 / top-5 overlap** counts how many of the baseline's leading barangays remain in the same leading set under a scenario. The fractions divide by 1, 3, and 5 respectively (or by the cohort size when it is smaller).
- **Top-3 Jaccard** is the size of the intersection divided by the size of the union of the two top-three sets.
- **Spearman rho** and **Kendall tau-b** compare the full ranking. Values closer to 1 mean the ordering is more similar to the baseline.
- **Mean / maximum absolute rank change** measure how far barangays move in rank positions.
- **Band agreement** is the percentage of barangays that stay in the same priority band (High Priority / Standard / Monitor).
- **Dispatch top-three** repeats the top-three overlap only among dispatch-eligible barangays, when the required columns are present.

A low overlap or a large rank change means the ranking is **sensitive to that weight choice**. It is not, by itself, evidence that one weighting is wrong. This notebook deliberately does **not** define a stable/unstable pass threshold. Report the distributions, averages, and worst cases, and interpret them with the policy context.

The analysis measures **ranking robustness under changed weights**. It does **not** prove predictive accuracy, violation probability, or better inspection outcomes.

## Optional, separate AHP consistency check

This cell is **optional and separate** from the OPS weight-sensitivity results. It does not change the baseline or any scenario.

If you have the actual AHP pairwise-comparison matrix, paste it into `AHP_MATRIX` below, or set `AHP_MATRIX_PATH` to a CSV/Excel file containing only the numeric matrix. The notebook will calculate:

- the principal-eigenvector weights
- `lambda_max`
- Consistency Index `CI = (lambda_max - n) / (n - 1)`
- the standard random index `RI`
- Consistency Ratio `CR = CI / RI`

No matrix is provided by default. The notebook will not invent one.

In [ ]:
# ============================================================
# OPTIONAL — AHP pairwise-matrix consistency (separate from OPS results)
# ============================================================
# Paste the actual matrix here as a list of lists. Leave as None to skip.
AHP_MATRIX = None

# Or point to a CSV/XLSX file containing only the numeric matrix.
AHP_MATRIX_PATH = None

RI_TABLE = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49,
    11: 1.51,
    12: 1.48,
    13: 1.56,
    14: 1.57,
    15: 1.59,
}

if AHP_MATRIX is None and AHP_MATRIX_PATH is None:
    print(
        "AHP check skipped. Supply AHP_MATRIX or AHP_MATRIX_PATH if you want the "
        "consistency calculation. No matrix was invented."
    )
else:
    if AHP_MATRIX is not None and AHP_MATRIX_PATH is not None:
        raise ValueError("Set only one of AHP_MATRIX or AHP_MATRIX_PATH, not both.")

    if AHP_MATRIX_PATH is not None:
        ahp_path = Path(AHP_MATRIX_PATH).expanduser().resolve()
        if not ahp_path.exists():
            raise FileNotFoundError(f"AHP_MATRIX_PATH does not exist: {ahp_path}")
        if ahp_path.suffix.lower() == ".csv":
            ahp_df = pd.read_csv(ahp_path, header=None)
        elif ahp_path.suffix.lower() == ".xlsx":
            ahp_df = pd.read_excel(ahp_path, header=None, engine="openpyxl")
        else:
            raise ValueError("AHP_MATRIX_PATH must be a .csv or .xlsx file.")
        matrix = ahp_df.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    else:
        matrix = np.asarray(AHP_MATRIX, dtype=float)

    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f"The AHP matrix must be square. Received shape {matrix.shape}.")
    if matrix.shape[0] < 1:
        raise ValueError("The AHP matrix is empty.")
    if not np.isfinite(matrix).all():
        raise ValueError("The AHP matrix contains blank, non-numeric, or infinite values.")
    if (matrix <= 0).any():
        raise ValueError("Every AHP pairwise entry must be greater than zero.")

    n_criteria = matrix.shape[0]
    problems = []
    if not np.allclose(np.diag(matrix), 1.0, atol=1e-6):
        problems.append("Diagonal entries must equal 1.")
    if not np.allclose(matrix * matrix.T, 1.0, atol=1e-6):
        problems.append("The matrix must be reciprocal: a_ij * a_ji = 1.")
    if problems:
        print("AHP matrix validation failed:")
        for problem in problems:
            print(f"  - {problem}")
        raise ValueError("The supplied AHP matrix is not a valid pairwise-comparison matrix.")

    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    real_eigenvalues = eigenvalues.real
    principal_index = int(np.argmax(real_eigenvalues))
    lambda_max = float(real_eigenvalues[principal_index])
    weights = np.abs(eigenvectors[:, principal_index].real)
    weights = weights / weights.sum()

    if n_criteria == 1:
        CI = 0.0
        CR = 0.0
        RI = 0.0
    else:
        CI = (lambda_max - n_criteria) / (n_criteria - 1)
        RI = RI_TABLE.get(n_criteria)
        CR = (CI / RI) if (RI is not None and RI > 0) else None

    ahp_weights = pd.DataFrame(
        {
            "criterion": [f"Criterion {index + 1}" for index in range(n_criteria)],
            "weight": weights,
            "weight_percent": weights * 100.0,
        }
    )

    print("AHP principal-eigenvector weights:")
    display(ahp_weights)
    print(f"lambda_max: {lambda_max:.6f}")
    print(f"CI: {CI:.6f}")
    print(f"RI: {RI if RI is not None else 'not available for n > 15'}")
    print(f"CR: {CR:.6f}" if CR is not None else "CR: not available")
    print(
        "\nA common AHP convention is CR < 0.10, but that is a convention for the "
        "matrix you supplied, not a REVELA stability threshold. This calculation is "
        "separate from the OPS weight-sensitivity results and does not change them."
    )

    AHP_RESULTS = {
        "matrix": matrix,
        "n": n_criteria,
        "weights": weights,
        "lambda_max": lambda_max,
        "CI": CI,
        "RI": RI,
        "CR": CR,
    }

## Before interpreting the results

Verify all of the following against the production export:

1. **Cohort:** one row per barangay in the full municipality. No dashboard filter, date filter, or barangay display filter was applied.
2. **Counts:** distinct eligible geospatial records, not unique businesses. Counts use the production **effective** colour. The export did not use a manually simplified `WHERE flagColor IN (...)` filter.
3. **Black count:** the `black_count` column is present. A zero means the exported production data contains zero effective Black records, not that Black was assumed absent.
4. **Yellow count:** if `yellow_count` is zero, the Yellow contribution could not be evaluated. The notebook states this limitation prominently.
5. **Mean coordinates:** `avg_lat`/`avg_lng` are the mean coordinates of valid, **dispatchable effective Red/Yellow** records. They are not the barangay centroid.
6. **BPLO:** the coordinates match the production WLC configuration in use for the same snapshot.
7. **Sector scores:** the configured barangay scores are supplied, or the no-override fallback of 50 is explicitly confirmed. A neutral 50 is not a measured industry-risk score.
8. **Baseline weights:** production scoring reads `revela_backend/api/analytics/wlc_config.json`, not the `wlc_config` database table. The JSON file ships with **68 / 7 / 25** and is what the Analytics page displays and scores with. The database table is legacy: its stored **40 / 40 / 20** values are not read anywhere in the scoring path, so they affect neither the displayed weights nor the OPS values. The export script records both values in its `.meta.json`. If the deployed JSON ever differs from 68/7/25, the notebook's parity check fails and the grid does not run.
9. **Parity:** if `production_ops` and `production_rank` were supplied, the maximum absolute OPS difference must be at most 0.1. If they were not supplied, parity is **NOT CHECKED**.
10. **Dispatch:** the dispatch-eligible comparison is shown only when all four eligibility columns were supplied. It uses dispatchable Red/Yellow records, dispatchable businesses, and the open-inspection exclusion.
11. **AHP:** the optional AHP matrix is the real matrix you supplied. Its consistency calculation is separate from the OPS sensitivity results.

### Required columns (short list)

`barangay_id`, `barangay_name`, `red_count`, `yellow_count`, `black_count`, `avg_lat`, `avg_lng`

Plus one of:

- `sector_score`, or `SECTOR_OVERRIDES_ABSENT = True` after confirming no overrides exist
- `bplo_lat` and `bplo_lng`, or `BPLO_LAT` / `BPLO_LNG` entered from production

Optional but needed for specific checks:

- parity: `production_ops`, `production_rank` (both)
- dispatch top-three: `dispatchable_red_count`, `dispatchable_yellow_count`, `dispatchable_business_count`, `open_inspection_excluded` (all four)
- cross-check: `flagged_count`

### Scope

This notebook measures **ranking robustness under changed weights**. It does not prove predictive accuracy, does not estimate violation probability, and does not show that one weighting produces better inspection outcomes.